In [ ]:
!uv sync --package simpler-mcp

# Multimodal MCPs Are Simpler Than You Think

## Let other agents search your data

You have photos and video. Other people have agents. An MCP server gives those agents named tools for searching your catalog.

We will build four recipes. Each recipe adds a kind of search to Pixeltable, proves it in this notebook, then exposes it through MCP. The server can search and answer. It cannot add media, drop tables, or run arbitrary Python.

This tutorial was inspired by Miguel Otero Pedrido's [How real AI Engineers build MCP Servers](https://theneuralmaze.substack.com/p/how-real-ai-engineers-build-mcp-servers), Alex Razvant's [Building MCP Servers for Video Search Engines](https://read.theaimerge.com/p/building-mcp-servers-for-video-search), and Avi Chawla's [Build the Ultimate MCP Server for Multimodal AI](https://blog.dailydoseofds.com/p/build-the-ultimate-mcp-server-for/).

In [ ]:
import atexit
import os
from pathlib import Path
import socket
import subprocess
import time

import mcp
from mcp.client.streamable_http import streamablehttp_client
import pixeltable as pxt

MCP_URL = 'http://127.0.0.1:8000/mcp'
server_process = None


def restart_server():
    """Restart FastMCP so its tool list matches the current schema."""
    global server_process
    if server_process is not None:
        server_process.terminate()
        server_process.wait(timeout=10)
    server_process = subprocess.Popen(
        ['uv', 'run', '--package', 'simpler-mcp', 'python', 'server.py'],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.STDOUT,
    )
    for _ in range(60):
        if server_process.poll() is not None:
            raise RuntimeError('The MCP server stopped during startup.')
        try:
            with socket.create_connection(('127.0.0.1', 8000), timeout=1):
                return
        except OSError:
            time.sleep(1)
    raise TimeoutError('The MCP server did not start within 60 seconds.')


async def list_mcp_tools():
    async with streamablehttp_client(MCP_URL) as (read, write, _):
        async with mcp.ClientSession(read, write) as session:
            await session.initialize()
            return [tool.name for tool in (await session.list_tools()).tools]


async def call_mcp_tool(name, arguments):
    async with streamablehttp_client(MCP_URL) as (read, write, _):
        async with mcp.ClientSession(read, write) as session:
            await session.initialize()
            return await session.call_tool(name, arguments)


@atexit.register
def stop_server():
    if server_process is not None:
        server_process.terminate()


# Uncomment this once when you want a clean catalog, then run from Recipe 1.
# pxt.drop_dir('mcp_demo', force=True)

## Recipe 1: search titles and captions

Start with the words on each photo. Human titles are short. BLIP adds a caption as a computed column.

Run the search in Pixeltable first. Then start the MCP and call the same search as a tool.

In [ ]:
print(Path('recipe1_store.py').read_text())
!pxt schema diff recipe1_store.py mcp_demo
!pxt schema update recipe1_store.py mcp_demo -f

BASE = 'https://raw.githubusercontent.com/pixeltable/pixeltable/main/docs/resources/images/'
photos = pxt.get_table('mcp_demo.photos')
if photos.count() == 0:
    photos.insert([
        {'photo_id': 1, 'image': BASE + '000000000001.jpg', 'title': 'Taxi on a city street'},
        {'photo_id': 2, 'image': BASE + '000000000025.jpg', 'title': 'Giraffe at the zoo'},
        {'photo_id': 3, 'image': BASE + '000000000034.jpg', 'title': 'Zebra grazing'},
        {'photo_id': 4, 'image': BASE + '000000000061.jpg', 'title': 'Elephant trek in the jungle'},
        {'photo_id': 5, 'image': BASE + '000000000080.jpg', 'title': 'Zebra in an enclosure'},
        {'photo_id': 6, 'image': BASE + '000000000090.jpg', 'title': 'Cow in a pasture'},
    ])

# Pixeltable search
photos.where(photos.search_title.contains('zebra')).select(
    photos.photo_id, photos.image, photos.title, photos.caption
).collect()

In [ ]:
# Caption search still needs the same word.
photos.where(photos.caption.lower().contains('striped')).select(
    photos.photo_id, photos.image, photos.title, photos.caption
).collect()

# MCP search
restart_server()
await list_mcp_tools(), await call_mcp_tool(
    'search_photo_text', {'query': 'zebra', 'limit': 5}
)

## Recipe 2: search image content

Words miss a photo when the title and caption use different language. CLIP compares the question with the pixels.

Apply the index, run the search in Pixeltable, then restart the server. The new MCP tool appears because the index now exists.

In [ ]:
print(Path('recipe2_search.py').read_text())
!pxt schema diff recipe2_search.py mcp_demo
!pxt schema update recipe2_search.py mcp_demo -f

photos = pxt.get_table('mcp_demo.photos')
score = photos.image.similarity(string='a striped animal', idx='clip_idx')
photos.order_by(score, asc=False).select(
    photos.photo_id, photos.image, photos.title, score=score
).limit(3).collect()

restart_server()
await list_mcp_tools(), await call_mcp_tool(
    'search_photo_content', {'query': 'a striped animal', 'limit': 5}
)

## Recipe 3: search video

A video search starts with frames. Pixeltable samples each video, captions the frames, and keeps a CLIP index on them.

Search the frames in Pixeltable first. The MCP then adds two ways to get a short clip. One starts with text. The other starts with an image.

In [ ]:
print(Path('recipe3_video.py').read_text())
!pxt schema diff recipe3_video.py mcp_demo
!pxt schema update recipe3_video.py mcp_demo -f

VIDEO_BASE = (
    'https://github.com/pixeltable/pixeltable/raw/release/'
    'docs/resources/audio-transcription-demo/'
)
videos = pxt.get_table('mcp_demo.videos')
if videos.count() == 0:
    videos.insert([
        {
            'video_id': 1,
            'video': VIDEO_BASE + 'Lex-Fridman-Podcast-430-Excerpt-0.mp4',
            'title': 'Lex Fridman Podcast 430, excerpt 0',
        },
        {
            'video_id': 2,
            'video': VIDEO_BASE + 'Lex-Fridman-Podcast-430-Excerpt-1.mp4',
            'title': 'Lex Fridman Podcast 430, excerpt 1',
        },
    ])

frames = pxt.get_table('mcp_demo.video_frames')
scene_score = frames.frame.similarity(string='two people talking in a studio', idx='frame_clip_idx')
frames.order_by(scene_score, asc=False).select(
    frames.video_id,
    frames.title,
    frames.frame,
    frames.caption,
    frames.clip,
    time_seconds=frames.frame_attrs.time,
    score=scene_score,
).limit(3).collect()

In [ ]:
restart_server()
text_clip = await call_mcp_tool(
    'get_clip_for_text', {'query': 'two people talking in a studio'}
)
text_clip

# Use one stored frame as the image query.
query_frame = frames.select(frames.frame).limit(1).collect().to_pandas().iloc[0]['frame']
image_clip = await call_mcp_tool(
    'get_clip_for_image', {'image': str(query_frame)}
)
image_clip

## Recipe 4: answer from captions and transcripts

Frames tell you what was on screen. Audio tells you what people said. Pixeltable extracts the audio and runs a local Whisper model.

Search captions and transcripts in Pixeltable. The final MCP tool retrieves from both indexes, then stores and returns an answer. It needs `OPENAI_API_KEY` for the answer column.

In [ ]:
print(Path('recipe4_transcript.py').read_text())
!pxt schema diff recipe4_transcript.py mcp_demo
!pxt schema update recipe4_transcript.py mcp_demo -f

videos = pxt.get_table('mcp_demo.videos')
frames = pxt.get_table('mcp_demo.video_frames')
question = 'What do they say about intelligence?'

speech_score = videos.transcript.similarity(string=question, idx='transcript_idx')
videos.order_by(speech_score, asc=False).select(
    videos.video_id, videos.title, videos.transcript, score=speech_score
).limit(2).collect()

In [ ]:
restart_server()
if os.getenv('OPENAI_API_KEY'):
    answer = await call_mcp_tool(
        'ask_about_video', {'question': question, 'limit': 3}
    )
    answer
else:
    print('Set OPENAI_API_KEY where the Pixeltable daemon runs, then rerun this cell.')

# The data owner can inspect every MCP search and every stored answer.
queries = pxt.get_table('mcp_demo.queries')
answers = pxt.get_table('mcp_demo.answers')
queries.order_by(queries.created_at, asc=False).select(
    queries.tool, queries.text, queries.image, queries.created_at
).limit(10).collect(), answers.select(
    answers.question, answers.source_notes, answers.answer, answers.created_at
).collect()

## Keep the data boundary

The final MCP can search photos, return video clips, and answer from captions and transcripts. It also stores tool calls in `queries` for the data owner.

The server has no tool for adding media or dropping tables. Those operations stay with the data owner.

In [ ]:
# The final tool list contains no drop or media-ingest tool.
await list_mcp_tools()

stop_server()